# Fine-tune LFM2.5-8B-A1B with Halo

This notebook fine-tunes [LFM2.5-8B-A1B](https://huggingface.co/LiquidAI/LFM2.5-8B-A1B) on UltraChat 200K.

Halo distributes the model's 32 routed experts across two GPUs. Each process owns 16 experts during training.

Halo gathers the expert weights when it saves. The result is a standard Hugging Face checkpoint.


## Configuration

This recipe uses full-parameter BF16 training with two-way expert parallelism.

| Setting | Value |
| --- | --- |
| GPUs | 2 |
| Expert parallel size | 2 |
| Dataset | UltraChat 200K, supervised split |
| Sequence length | 8,192 |
| Effective batch size | 16 |
| Output | Gathered Hugging Face checkpoint |

Do not enable context parallelism. LFM2 short-convolution layers operate across the sequence axis.


## Start the Halo notebook server

This notebook uses the public Halo image that matches the detected GPUs. It needs two matching NVIDIA GPUs and a large data volume.

Run these commands on the host. Replace the three paths before you start the container.

~~~bash
git clone --recurse-submodules https://github.com/whitecircle/halo.git
git clone https://github.com/Liquid4All/cookbook.git

export HALO_DIR=/path/to/halo
export COOKBOOK_DIR=/path/to/cookbook
export DATA_DIR=/path/to/large/volume
GPU_NAME="$(nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1)"
case "$GPU_NAME" in
  *H100*|*H200*) HALO_IMAGE=public.ecr.aws/whitecircle/halo:hopper ;;
  *B200*|*B300*|*GB200*|*GB300*) HALO_IMAGE=public.ecr.aws/whitecircle/halo:blackwell ;;
  *) echo "Unsupported GPU: $GPU_NAME"; exit 1 ;;
esac
export HALO_IMAGE

docker pull "$HALO_IMAGE"
docker run --rm -it --gpus '"device=0,1"' \
  --network host \
  --ipc=host --shm-size=128g \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -e HF_HOME=/mnt/hf \
  -e HF_DATASETS_CACHE=/mnt/hf/datasets \
  -e TMPDIR=/mnt/tmp \
  -e HALO_DATA_ROOT=/mnt \
  -e PYTHONPATH=/workspace \
  -e CUDA_DEVICE_MAX_CONNECTIONS=1 \
  -v "$HALO_DIR":/workspace \
  -v "$COOKBOOK_DIR":/cookbook \
  -v "$DATA_DIR":/mnt \
  -w /workspace \
  "$HALO_IMAGE" \
  jupyter lab --ip=0.0.0.0 --port=8888 --no-browser \
    --allow-root --notebook-dir=/cookbook/finetuning/notebooks
~~~

Open the URL printed by Jupyter. Then open this notebook.


## Check the runtime

The notebook does not create a CUDA context before the distributed launch.


In [ ]:
import subprocess
from pathlib import Path

RUN_ROOT = Path("/mnt/checkpoints")
CONFIG_PATH = RUN_ROOT / "lfm25-8b-a1b-ultrachat-ep2.yaml"
OUTPUT_PATH = RUN_ROOT / "lfm25-8b-a1b-ultrachat-ep2"

gpu_names = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name",
        "--format=csv,noheader",
    ],
    text=True,
).strip().splitlines()
assert len(gpu_names) == 2, f"Expected 2 visible GPUs, found {len(gpu_names)}"

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(gpu_names)


## Create the training configuration

This configuration matches Halo's validated LFM2 MoE recipe.


In [ ]:
CONFIG_PATH.write_text(
    r"""
model_name_or_path: LiquidAI/LFM2.5-8B-A1B
moe_balancing: bias_update

dataset:
- HuggingFaceH4/ultrachat_200k@train_sft
conversation_field: messages
test_size: 0.01
train_only_on_completions: true
assistant_message_template: "<|im_start|>assistant\n"
pad_token: "<|pad|>"
eos_token: "<|im_end|>"

expert_parallel_size: 2
save_sharded_ep: false
use_grouped_gemm: true
fp32_router: true
fp32_experts: false

attn_implementation: flash_attention_2
use_liger_kernel: false
packing: true
max_length: 8192
bf16: true

per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8
num_train_epochs: 1.0
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false

optim: adamw_torch_fused
learning_rate: 5.0e-06
lr_scheduler_type: cosine
warmup_steps: 32
max_grad_norm: 1.0

save_strategy: steps
save_steps: 1000
eval_strategy: steps
eval_steps: 300
save_total_limit: 1
save_only_model: true
output_dir: /mnt/checkpoints/lfm25-8b-a1b-ultrachat-ep2

logging_steps: 1
logging_first_step: true
report_to: none
remove_unused_columns: false
dataloader_num_workers: 2

use_peft: false
""".lstrip()
)
print(CONFIG_PATH.read_text())


Halo uses DeepEP for token dispatch and grouped GEMM for expert computation.

The router runs in FP32. The experts run in BF16.

The bias update balances expert selection without a router auxiliary loss.


## Check the launch command

The dry run prints the two-process command without loading the model.


In [ ]:
!halo launch sft {CONFIG_PATH} -n 2 --dry-run


## Start training

This cell starts the full UltraChat run on both visible GPUs.


In [ ]:
!halo launch sft {CONFIG_PATH} -n 2


## Load the saved checkpoint

The saved checkpoint uses the native Transformers format.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_PATH)
model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
)

messages = [
    {
        "role": "user",
        "content": "Summarize this support ticket and list the next actions.",
    }
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)
output = model.generate(
    inputs,
    max_new_tokens=256,
    do_sample=False,
)
reply = tokenizer.decode(
    output[0, inputs.shape[-1] :],
    skip_special_tokens=True,
)
print(reply)
